# Протей на TinyStoriesОдин вечер до ответа: заговорит ли модель.Порядок: GPU → код → данные → проба 200 шагов → полный прогон.**Не пропускайте пробу** — если loss не падает за 200 шагов, ждать 4 часа незачем.

## 1. GPU и Drive

In [ ]:
import torchassert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"print(torch.cuda.get_device_name(0))from google.colab import drivedrive.mount('/content/drive')

## 2. Код и зависимости

In [ ]:
!pip install -q datasets!git clone -q https://github.com/maleshovivan23-creator/-.git /content/ultranet || true%cd /content/ultranet!git checkout -q arena/01a0b93c-repo && git pull -qimport sys; sys.path.insert(0, '/content/ultranet')

## 3. ДанныеПервый запуск: ~10 минут. Для пробы поставьте `--limit 50000`.

In [ ]:
!python colab/prepare_data.py --vocab 4096

## 4. Проба: 200 шаговСмотрим только на одно — падает ли loss. Должен уйти с ~8 до ~5.

In [ ]:
!python colab/train.py --steps 200 --out /content/drive/MyDrive/proteus/probe

## 5. Полный прогонЧекпоинт каждые 500 шагов на Drive — Colab рвёт сессию через ~12 часов,и без этого прогон потеряется. Эта же ячейка **продолжит с места обрыва**,если её перезапустить.

In [ ]:
!python colab/train.py \    --steps 60000 --batch 32 --seq 256 \    --dim 256 --layers 6 --heads 8 --lr 6e-4 \    --out /content/drive/MyDrive/proteus/tinystories-16m

## 6. Сэмплы

In [ ]:
import torch, syssys.path.insert(0, '/content/ultranet'); sys.path.insert(0, '/content/ultranet/colab')from ultranet.models import GPTConfigfrom ultranet.torch_port import TorchGPTfrom ultranet.tokenizer import BPETokenizerfrom train import sampleOUT = '/content/drive/MyDrive/proteus/tinystories-16m'tok = BPETokenizer.load('data/tokenizer.json')st = torch.load(f'{OUT}/last.pt', map_location='cuda', weights_only=False)model = TorchGPT(GPTConfig(**st['gcfg'])).cuda(); model.load_state_dict(st['model'])print(f"шаг {st['step']}")for p in ["Once upon a time", "Lily found a", "The little boy was very"]:    print('>', sample(model, tok, p, 120, 'cuda', st['gcfg']['block_size'], temperature=0.8))    print()

## 7. Кривая обучения

In [ ]:
import json, matplotlib.pyplot as pltrows=[json.loads(l) for l in open(f'{OUT}/log.jsonl')]plt.figure(figsize=(9,4))plt.plot([r['step'] for r in rows],[r['loss'] for r in rows])plt.xlabel('шаг'); plt.ylabel('loss'); plt.grid(alpha=.3); plt.show()print(f"{rows[-1]['tok_s']:,} токенов/с")